# dim_indicators: Gold

Builds `pondolens.dev_gold.dim_indicator`. `FACT_REGIONAL_INDICATOR` looks up `indicator_id` here.

| indicator | Silver source | measure_type | unit |
|---|---|---|---|
| `unemployment` | `pondolens.dev_silver.unemployment_rate` | `unemployment_rate` | `% of labor force` |
| `grdp` | `pondolens.dev_silver.grdp_per_capita` | `grdp_per_capita` (one row per `price_basis`) | `pesos` |
| `service_access` | `pondolens.dev_silver.service_access` | from `service_measure` | as in Silver (`% of population`, `% of households`, …) |

**Rules**
- **Business key:** `indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value`. One row per unique combination; the MERGE matches on it.
- **`indicator_id`:** range **40001–49999** (team rule: DIM_INDICATOR = 40000). Existing indicators keep their ID; only new ones get `MAX(indicator_id) + ROW_NUMBER()`.
- **Not applicable → `'N/A'`** (never NULL), so dashboard filters are clean and the fact table can join with plain `=`.
- **`price_basis`** (GRDP only): `'2018'` = constant 2018 prices; `'2021'`–`'2025'` = current prices of that year.
- Unemployment and GRDP are regional totals: `breakdown_type = 'Total'`, `breakdown_value = 'All'`.
- Service-access units stay exactly as in Silver; the team keeps different percentage bases separate.
- **Indicators that disappear from Silver are kept**, because fact rows may still reference them.
- **Idempotent:** rerunning with unchanged Silver inserts 0 rows.

**No duplicate IDs, even if two people run this at the same time (no run lock)**
1. IDs are calculated **inside the MERGE**, so reading the current maximum and inserting happen in one atomic transaction.
2. The table uses Delta **`Serializable` isolation**. If another run commits while this one is in progress, Delta rejects this run's commit (`ConcurrentAppendException`) instead of saving IDs based on an outdated maximum.
3. A check after the MERGE confirms IDs and business keys are unique.

If you ever see `ConcurrentAppendException`, another run finished first. Nothing was saved by this run; **just run it again** and it will continue from the new maximum.

Requires Databricks SQL or DBR 14.3+. Run all cells top to bottom.

## 1. Create the Gold table

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pondolens.dev_gold;

CREATE TABLE IF NOT EXISTS pondolens.dev_gold.dim_indicator (
  indicator_id INT NOT NULL COMMENT 'Surrogate key, 40001-49999',
  indicator STRING NOT NULL COMMENT 'unemployment, grdp, service_access',
  measure_type STRING NOT NULL COMMENT 'unemployment_rate, grdp_per_capita, electricity_access, ...',
  price_basis STRING NOT NULL COMMENT '2018 = constant 2018 prices; 2021-2025 = current prices; N/A otherwise',
  unit STRING NOT NULL COMMENT '% of labor force, pesos, % of population, % of households, ...',
  service STRING NOT NULL COMMENT 'Water, Electricity, Internet; N/A otherwise',
  breakdown_type STRING NOT NULL,
  breakdown_value STRING NOT NULL,
  CONSTRAINT dim_indicator_pk PRIMARY KEY (indicator_id)
)
USING DELTA
TBLPROPERTIES ('delta.isolationLevel' = 'Serializable')
COMMENT 'Gold indicator dimension. Business key: indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value.';

-- Makes sure a table created by an earlier version also uses Serializable isolation.
ALTER TABLE pondolens.dev_gold.dim_indicator SET TBLPROPERTIES ('delta.isolationLevel' = 'Serializable');

## 2. Build the indicator list from Silver and check it

Each Silver table is reduced to its distinct indicators first, so only a handful of rows reach the MERGE.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dim_indicator_stage AS
SELECT DISTINCT
  'unemployment' AS indicator, 'unemployment_rate' AS measure_type, 'N/A' AS price_basis,
  '% of labor force' AS unit, 'N/A' AS service, 'Total' AS breakdown_type, 'All' AS breakdown_value
FROM pondolens.dev_silver.unemployment_rate
UNION
SELECT DISTINCT
  'grdp', 'grdp_per_capita', CAST(price_basis AS STRING),
  'pesos', 'N/A', 'Total', 'All'
FROM pondolens.dev_silver.grdp_per_capita
UNION
SELECT DISTINCT
  'service_access',
  LOWER(TRIM(BOTH '_' FROM REGEXP_REPLACE(TRIM(service_measure), '[^A-Za-z0-9]+', '_'))),  -- 'Electricity access' -> 'electricity_access'
  'N/A',
  COALESCE(NULLIF(TRIM(unit), ''), 'N/A'),
  COALESCE(NULLIF(TRIM(service), ''), 'N/A'),
  COALESCE(NULLIF(TRIM(breakdown_type), ''), 'N/A'),
  COALESCE(NULLIF(TRIM(breakdown_value), ''), 'N/A')
FROM pondolens.dev_silver.service_access;

SELECT
  ASSERT_TRUE(COUNT(*) > 0, 'No indicators found in Silver.') AS has_rows,
  ASSERT_TRUE(COUNT_IF(NULLIF(measure_type, '') IS NULL OR NULLIF(price_basis, '') IS NULL) = 0,
    'Blank measure_type or price_basis. Check the Silver source.') AS keys_present,
  ASSERT_TRUE(COUNT_IF(indicator = 'grdp' AND price_basis NOT RLIKE '^[0-9]{4}$') = 0,
    'GRDP price_basis must be a 4-digit year.') AS valid_price_basis
FROM dim_indicator_stage;

-- Existing Gold must already have unique IDs, unique keys and IDs in range.
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(DISTINCT indicator_id)
  AND COUNT(*) = COUNT(DISTINCT STRUCT(indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value))
  AND COUNT_IF(indicator_id NOT BETWEEN 40001 AND 49999) = 0,
  'Gold has duplicate IDs, duplicate keys or IDs outside 40001-49999.'
)
FROM pondolens.dev_gold.dim_indicator;

SELECT * FROM dim_indicator_stage ORDER BY indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value;

## 3. Merge new indicators (IDs assigned inside the MERGE)

- `ROW_NUMBER()` runs only on indicators not yet in Gold, continuing from the current maximum (40000 when empty).
- Every column except `indicator_id` is part of the business key, so matched rows have nothing to update: the MERGE only inserts.
- An ID above 49999 stops the MERGE before anything is written.

In [0]:
%sql
MERGE INTO pondolens.dev_gold.dim_indicator AS t
USING (
  SELECT
    n.*,
    CASE WHEN n.new_id <= 49999 THEN CAST(n.new_id AS INT)
         ELSE CAST(RAISE_ERROR('indicator_id would exceed 49999 (FACT_INDICATORS range starts at 50000).') AS INT)
    END AS indicator_id
  FROM (
    SELECT s.*, l.last_id + ROW_NUMBER() OVER (ORDER BY s.indicator, s.measure_type, s.price_basis, s.unit, s.service, s.breakdown_type, s.breakdown_value) AS new_id
    FROM dim_indicator_stage AS s
    LEFT ANTI JOIN pondolens.dev_gold.dim_indicator AS d
      ON s.indicator = d.indicator
      AND s.measure_type = d.measure_type
      AND s.price_basis = d.price_basis
      AND s.unit = d.unit
      AND s.service = d.service
      AND s.breakdown_type = d.breakdown_type
      AND s.breakdown_value = d.breakdown_value
    CROSS JOIN (SELECT COALESCE(MAX(indicator_id), 40000) AS last_id FROM pondolens.dev_gold.dim_indicator) AS l
  ) AS n
) AS s
ON t.indicator = s.indicator
  AND t.measure_type = s.measure_type
  AND t.price_basis = s.price_basis
  AND t.unit = s.unit
  AND t.service = s.service
  AND t.breakdown_type = s.breakdown_type
  AND t.breakdown_value = s.breakdown_value
WHEN NOT MATCHED THEN INSERT (indicator_id, indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value)
VALUES (s.indicator_id, s.indicator, s.measure_type, s.price_basis, s.unit, s.service, s.breakdown_type, s.breakdown_value);

## 4. Post-merge checks and results

Final safety net: IDs and business keys are unique, and every Silver indicator is in Gold.

In [0]:
%sql
SELECT
  ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT indicator_id) AND COUNT(*) = COUNT(DISTINCT STRUCT(indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value)),
    'Duplicate IDs or business keys after the merge.') AS unique_ids_and_keys
FROM pondolens.dev_gold.dim_indicator;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Some Silver indicators are missing from Gold.')
FROM dim_indicator_stage AS s
LEFT ANTI JOIN pondolens.dev_gold.dim_indicator AS t
  ON s.indicator = t.indicator
  AND s.measure_type = t.measure_type
  AND s.price_basis = t.price_basis
  AND s.unit = t.unit
  AND s.service = t.service
  AND s.breakdown_type = t.breakdown_type
  AND s.breakdown_value = t.breakdown_value;

-- Indicators no longer in Silver (kept on purpose; normally empty).
SELECT t.*
FROM pondolens.dev_gold.dim_indicator AS t
LEFT ANTI JOIN dim_indicator_stage AS s
  ON t.indicator = s.indicator
  AND t.measure_type = s.measure_type
  AND t.price_basis = s.price_basis
  AND t.unit = s.unit
  AND t.service = s.service
  AND t.breakdown_type = s.breakdown_type
  AND t.breakdown_value = s.breakdown_value;

SELECT * FROM pondolens.dev_gold.dim_indicator ORDER BY indicator_id;

## For FACT_REGIONAL_INDICATOR (lookup example)

```sql
-- Unemployment
SELECT d.indicator_id, u.year, u.region_code, u.unemployment_rate AS measurement
FROM pondolens.dev_silver.unemployment_rate AS u
JOIN pondolens.dev_gold.dim_indicator AS d ON d.indicator = 'unemployment';

-- GRDP per capita (one indicator per price_basis)
SELECT d.indicator_id, g.year, g.region_code, g.grdp_per_capita AS measurement
FROM pondolens.dev_silver.grdp_per_capita AS g
JOIN pondolens.dev_gold.dim_indicator AS d ON d.indicator = 'grdp' AND d.price_basis = CAST(g.price_basis AS STRING);
```

For service access, apply the same `measure_type`, `unit`, `service` and breakdown cleaning as in step 2, then join on all seven key columns.